In [3]:
import pandas as pd
file_path = "/home/u20240075/luoyashuo/G-KD/data/OmniThought-R1-0528.jsonl"
df = pd.read_json(file_path, lines=True)

In [4]:
shuffled_df = df.sample(frac=1, random_state=42)
shuffled_df = shuffled_df.reset_index(drop=True)
first_row = shuffled_df.iloc[0]

# 打印结果s
print(first_row["reasoning"][0]["Cognitive_Difficulty"]["level"])
print(type(first_row["reasoning"][0]["Cognitive_Difficulty"]["level"]))

7
<class 'int'>


In [8]:
import pandas as pd
from collections import defaultdict
import json

# 定义难度级别分组函数
def categorize_level(level):
    if 0 <= level <= 1:
        return '0-1'
    elif 2 <= level <= 3:
        return '2-3'
    elif 4 <= level <= 5:
        return '4-5'
    elif 6 <= level <= 7:
        return '6-7'
    elif 8 <= level <= 9:
        return '8-9'
    else:
        return '其他'

# 按难度级别对数据进行分组并计数
level_counts = defaultdict(int)
selected_rows = []

for _, row in shuffled_df.iterrows():
    try:
        level = row["reasoning"][0]["Cognitive_Difficulty"]["level"]
        level_group = categorize_level(level)
        
        # 若该难度组的数据不足 10000 条，则添加当前行
        if level_group != '其他' and level_counts[level_group] < 10000:
            sharegpt_entry = {
                "conversations": [
                    {
                        "from": "human",
                        "value": "Solve the following math problem: " + row["question"]
                    }
                ],
                "system": "You FIRST think about the reasoning process as an internal monologue and then provide the final answer. The reasoning process MUST BE enclosed within <think> </think> tags. The final answer MUST BE enclosed within <answer> </answer> tags."
            }
            selected_rows.append(sharegpt_entry)
            level_counts[level_group] += 1
            
            # 当所有难度组都收集满 10000 条数据时，停止遍历
            if all(count >= 10000 for count in level_counts.values() if count > 0):
                break
    except (KeyError, IndexError, TypeError) as e:
        # 处理数据格式异常情况
        print(f"处理行时出错: {e}")
        continue

# 将筛选出的数据转换为新的 DataFrame
balanced_df = pd.DataFrame(selected_rows)

# 输出各难度组的数据量
print("各难度级别选取的数据量:")
for group, count in level_counts.items():
    print(f"{group}: {count} 条")


# 保存为ShareGPT格式的JSONL文件
output_path = '/home/u20240075/luoyashuo/G-KD/data/balanced_data_sharegpt.jsonl'
with open(output_path, 'w', encoding='utf-8') as f:
    for entry in selected_rows:
        json_line = json.dumps(entry, ensure_ascii=False)
        f.write(json_line + '\n')

print(f"已保存 {len(selected_rows)} 条ShareGPT格式记录到 {output_path}")

各难度级别选取的数据量:
6-7: 10000 条
4-5: 10000 条
2-3: 10000 条
0-1: 385 条
8-9: 6847 条
已保存 37232 条ShareGPT格式记录到 /home/u20240075/luoyashuo/G-KD/data/balanced_data_sharegpt.jsonl


In [28]:
# # test
# import pandas as pd
# from collections import defaultdict

# # 读取本地 JSONL 文件
# file_path = "/home/u20240075/luoyashuo/G-KD/data/balanced_data.jsonl"
# df = pd.read_json(file_path, lines=True)

# # 定义难度级别分组函数
# def categorize_level(level):
#     if 0 <= level <= 1:
#         return '0-1'
#     elif 2 <= level <= 3:
#         return '2-3'
#     elif 4 <= level <= 5:
#         return '4-5'
#     elif 6 <= level <= 7:
#         return '6-7'
#     elif 8 <= level <= 9:
#         return '8-9'
#     else:
#         return '其他'

# # 初始化计数器
# level_counts = defaultdict(int)

# # 遍历DataFrame中的每一行
# for _, row in df.iterrows():
#     try:
#         # 提取认知难度值
#         cd_value = row["reasoning"][0]["Cognitive_Difficulty"]["level"]
        
#         # 获取难度类别
#         level_category = categorize_level(cd_value)
        
#         # 更新计数器
#         level_counts[level_category] += 1
#     except (KeyError, IndexError, TypeError) as e:
#         # 处理格式异常的样本
#         print(f"处理行时出错: {e}")
#         continue

# # 输出统计结果
# print("数据集中认知难度(cd)的分布统计:")
# for category, count in level_counts.items():
#     print(f"{category} 区间的样本数量: {count}")

数据集中认知难度(cd)的分布统计:
6-7 区间的样本数量: 10000
4-5 区间的样本数量: 10000
2-3 区间的样本数量: 10000
0-1 区间的样本数量: 385
8-9 区间的样本数量: 6847
